# BÀI THỰC HÀNH 4 – Phan Công Úy
## Frequent Pattern Mining & Association Rule Mining

**Mục tiêu:** hoàn thiện toàn bộ pipeline từ transaction data đến frequent itemsets, association rules, pattern evaluation và research question.

## Dataset

Sử dụng **Online Retail – UCI Machine Learning Repository**.

- Trang dữ liệu: https://archive.ics.uci.edu/dataset/352/online%2Bretail
- Tải `Online Retail.xlsx` và đặt cùng thư mục với notebook.
- `InvoiceNo` bắt đầu bằng `C` biểu thị giao dịch hủy.
- Trong bài: **Transaction = InvoiceNo**, **Item = Description** (có thể thảo luận dùng StockCode).


## 0. Chuẩn bị môi trường

In [26]:
# Chạy nếu môi trường chưa có thư viện
# !pip install mlxtend openpyxl -q

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import time

from mlxtend.frequent_patterns import apriori, fpgrowth, association_rules

pd.set_option("display.max_colwidth", 120)


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packag

In [27]:
def generate_rules(freq_itemsets, basket_df, min_conf=0.5):
    """Tương thích cả các phiên bản mlxtend mới/cũ."""
    try:
        return association_rules(
            freq_itemsets,
            num_itemsets=len(basket_df),
            metric="confidence",
            min_threshold=min_conf
        )
    except TypeError:
        return association_rules(
            freq_itemsets,
            metric="confidence",
            min_threshold=min_conf
        )

def normalize_itemset(s):
    return tuple(sorted(list(s)))

def rules_signature(df_rules):
    if df_rules is None or len(df_rules) == 0:
        return set()
    return set(
        (normalize_itemset(a), normalize_itemset(c))
        for a, c in zip(df_rules["antecedents"], df_rules["consequents"])
    )


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packag

## A. Đọc và khảo sát dữ liệu
**TODO:** đọc file, in shape/head/info/missing/nunique; viết nhận xét bằng Markdown.

In [ ]:
import pandas as pd

FILE = "Online Retail.xlsx"

# TODO 1: Đọc dữ liệu Excel vào biến df
df = pd.read_excel(FILE)

# TODO 2: Hiển thị shape, head, info, missing values và số unique
print("--- SHAPE OF DATAFRAME ---")
print(df.shape)

print("\n--- HEAD OF DATAFRAME ---")
print(df.head())

print("\n--- DATAFRAME INFO ---")
df.info()

print("\n--- MISSING VALUES IN EACH COLUMN ---")
print(df.isnull().sum())

print("\n--- NUMBER OF UNIQUE VALUES ---")
print(df.nunique())

Kết quả truyền trực tuyến bị cắt bớt đến 5000 dòng cuối.
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().

### Câu hỏi A

1. **Một dòng là gì?**
   - Mỗi dòng biểu diễn chi tiết của một mặt hàng cụ thể được bán ra trong một giao dịch (Invoice). Nó chứa thông tin như mã hóa đơn, mã sản phẩm, mô tả sản phẩm, số lượng, ngày bán, đơn giá, mã khách hàng, và quốc gia.

2. **Transaction ID là cột nào?**
   - Cột `InvoiceNo` đóng vai trò là Transaction ID (Mã giao dịch).

3. **Item nên dùng Description hay StockCode? Vì sao?**
   - Nên dùng `Description` nếu muốn kết quả trực quan và dễ đọc hiểu ngay lập tức cho các chiến lược kinh doanh. Tuy nhiên, dùng `StockCode` có ưu điểm là chuẩn hóa tốt hơn, tránh được trường hợp một sản phẩm có nhiều mô tả khác nhau (ví dụ: viết hoa/thường, lỗi chính tả). Trong bài toán này, ta sẽ dùng `Description` theo yêu cầu đề bài và thực hiện chuẩn hóa dữ liệu.

4. **Quantity và UnitPrice có ý nghĩa gì?**
   - `Quantity` là số lượng mặt hàng đó được mua trong giao dịch (nếu là số âm thì đại diện cho giao dịch bị hủy/trả hàng).
   - `UnitPrice` là đơn giá của một sản phẩm.

## B–C. Chuẩn hóa Item và Data Cleaning
**TODO:** chuẩn hóa Description; loại cancellation, Quantity ≤ 0, UnitPrice ≤ 0, Description rỗng. Ghi số dòng trước/sau.

In [29]:
print(f"Số dòng trước khi làm sạch: {len(df)}")

# TODO 3: Chuẩn hóa Description
df['Description'] = df['Description'].astype(str).str.strip().str.upper()

# TODO 4: Tạo cờ cancelled từ InvoiceNo
df['Cancelled'] = df['InvoiceNo'].astype(str).str.startswith('C')

# TODO 5: Tạo clean dataframe
clean_df = df[
    (~df['Cancelled']) &
    (df['Quantity'] > 0) &
    (df['UnitPrice'] > 0) &
    (df['Description'].notna()) &
    (df['Description'] != 'NAN') &
    (df['Description'] != '')
].copy()

# TODO 6: In báo cáo số dòng trước/sau
print(f"Số dòng sau khi làm sạch: {len(clean_df)}")
print(f"Số dòng bị loại bỏ: {len(df) - len(clean_df)}")

## D. Data Audit sau Cleaning

In [30]:
# TODO 7:
# - số rows
# - số invoices
# - số products
# - số countries
# - top 10 countries
# - top 15 products theo Quantity

n_rows = len(clean_df)
n_invoices = clean_df['InvoiceNo'].nunique()
n_products = clean_df['Description'].nunique()
n_countries = clean_df['Country'].nunique()

print(f"Số dòng (rows): {n_rows}")
print(f"Số hóa đơn (invoices): {n_invoices}")
print(f"Số sản phẩm (products): {n_products}")
print(f"Số quốc gia (countries): {n_countries}")

print("\n--- TOP 10 COUNTRIES BY TRANSACTION COUNT ---")
top_countries = clean_df['Country'].value_counts().head(10)
print(top_countries)

print("\n--- TOP 15 PRODUCTS BY TOTAL QUANTITY ---")
top_products = clean_df.groupby('Description')['Quantity'].sum().sort_values(ascending=False).head(15)
print(top_products)

## E–F. Chọn Country và tạo Transaction Basket
Mặc định dùng **France** để chạy trên lớp.

In [31]:
COUNTRY = "France"

# TODO 8: Lọc country_data
country_df = clean_df[clean_df['Country'] == COUNTRY]

# TODO 9: Tạo basket theo InvoiceNo x Description
basket_raw = country_df.groupby(['InvoiceNo', 'Description'])['Quantity'].sum().unstack().fillna(0)

# TODO 10: Binary hóa bằng > 0
basket = basket_raw.applymap(lambda x: x > 0)

# TODO 11: In shape và head
print(f"Hình dạng của ma trận giỏ hàng (Basket Shape): {basket.shape}")
print("\n--- 5 DÒNG ĐẦU TIÊN CỦA MA TRẬN GIỎ HÀNG ---")
print(basket.head())

### Câu hỏi F

1. **Tại sao Quantity=1 và Quantity=10 cùng trở thành True?**
   - Phép biến đổi nhị phân chỉ quan tâm đến sự xuất hiện của sản phẩm trong hóa đơn (có hay không có). Vì $1 > 0$ và $10 > 0$, cả hai trường hợp này đều được ánh xạ về `True` (nghĩa là sản phẩm xuất hiện trong giỏ hàng).

2. **Thông tin nào bị mất?**
   - Nhưng thông tin mất đi: về quy mô/số lượng mặt hàng được mua (`Quantity`) và gián tiếp mất đi thông tin về doanh thu, giá trị bằng tiền mà mặt hàng đó mang lại cho từng hóa đơn cụ thể.

3. **Nếu muốn giữ giá trị kinh tế thì cần mở rộng theo hướng nào?**
   - Cần áp dụng thuật toán khai thác mẫu phổ biến có trọng số (Weighted Association Rule Mining) hoặc thuật toán khai thác tập mục có lợi ích cao (High-Utility Itemset Mining - HUIM). Ở đó, mỗi mặt hàng sẽ được gán trọng số dựa trên số lượng (`Quantity`) kết hợp với đơn giá (`UnitPrice`) để tối ưu giá trị kinh tế/doanh thu.

## G. Tính Support bằng tay

In [32]:
# TODO 12: Chọn các mặt hàng phổ biến trong giỏ hàng để tính support thủ công
item1 = "POSTAGE"
item2 = "ROUND SNACK BOXES SET OF4 WOODLAND"

# Tính support của 1 item
sup_item1 = basket[item1].mean()
print(f"Support thủ công của {item1}: {sup_item1:.4f}")

# Tính support đồng xuất hiện của 2 items
sup_both = (basket[item1] & basket[item2]).mean()
print(f"Support đồng xuất hiện thủ công của [{item1}] và [{item2}]: {sup_both:.4f}")

## H. Apriori

In [33]:
import time
from mlxtend.frequent_patterns import apriori

MIN_SUPPORT = 0.05

# TODO 13: Chạy apriori và đo runtime
start_time = time.time()
apriori_itemsets = apriori(basket, min_support=MIN_SUPPORT, use_colnames=True)
runtime_apriori = time.time() - start_time

# Thêm cột length cho dễ phân tích
apriori_itemsets['length'] = apriori_itemsets['itemsets'].apply(lambda x: len(x))

# Sắp xếp giảm dần theo support
apriori_itemsets = apriori_itemsets.sort_values(by='support', ascending=False).reset_index(drop=True)

print(f"Thời gian chạy Apriori: {runtime_apriori:.4f} giây")
print(f"Số lượng tập phổ biến tìm được: {len(apriori_itemsets)}")
print("\n--- TOP 20 FREQUENT ITEMSETS (APRIORI) ---")
print(apriori_itemsets.head(20))

## I. Sensitivity theo min_support

In [34]:
import time
import pandas as pd
from mlxtend.frequent_patterns import apriori

supports = [0.10, 0.07, 0.05, 0.03, 0.02]
results = []

for sup in supports:
    start_time = time.time()
    freq_sets = apriori(basket, min_support=sup, use_colnames=True)
    elapsed = time.time() - start_time
    results.append({
        'min_support': sup,
        'num_patterns': len(freq_sets),
        'runtime_sec': elapsed
    })

sensitivity_df = pd.DataFrame(results)
print("--- SENSITIVITY TABLE ---")
print(sensitivity_df)

In [35]:
import matplotlib.pyplot as plt

fig, ax1 = plt.subplots(figsize=(8, 5))

# Vẽ số lượng pattern
color = 'tab:blue'
ax1.set_xlabel('Min Support')
ax1.set_ylabel('Number of Patterns', color=color)
ax1.plot(sensitivity_df['min_support'], sensitivity_df['num_patterns'], marker='o', color=color, linewidth=2)
ax1.tick_params(axis='y', labelcolor=color)
ax1.invert_xaxis()  # Giảm dần min_support để biểu đồ đi từ trái sang phải

# Vẽ runtime
ax2 = ax1.twinx()
color = 'tab:red'
ax2.set_ylabel('Runtime (seconds)', color=color)
ax2.plot(sensitivity_df['min_support'], sensitivity_df['runtime_sec'], marker='s', linestyle='--', color=color, linewidth=2)
ax2.tick_params(axis='y', labelcolor=color)

plt.title('Sensitivity Analysis: Min Support vs Number of Patterns & Runtime')
fig.tight_layout()
plt.show()

## J–K. Association Rules & Confidence Trap

In [36]:
from mlxtend.frequent_patterns import association_rules

def generate_rules(freq_itemsets, basket_df, min_conf=0.5):
    try:
        return association_rules(
            freq_itemsets,
            num_itemsets=len(basket_df),
            metric="confidence",
            min_threshold=min_conf
        )
    except TypeError:
        return association_rules(
            freq_itemsets,
            metric="confidence",
            min_threshold=min_conf
        )

rules = generate_rules(apriori_itemsets, basket, min_conf=0.5)
columns_to_keep = ['antecedents', 'consequents', 'support', 'confidence', 'lift']
filtered_rules = rules[columns_to_keep].sort_values(by='lift', ascending=False).reset_index(drop=True)

print(f"Tong so luat ket hop tim duoc: {len(filtered_rules)}")
print("\n--- TOP 20 LUAT KET HOP SAP XEP THEO LIFT ---")
print(filtered_rules.head(20))

In [37]:
confidence_trap_rules = filtered_rules[
    (filtered_rules['confidence'] >= 0.70) &
    (filtered_rules['lift'] >= 0.8) &
    (filtered_rules['lift'] <= 1.25)
]

print("--- CAC LUAT GAP BAY DO TIN CAY (CONFIDENCE TRAP) ---")
if len(confidence_trap_rules) > 0:
    print(confidence_trap_rules.head(10))
else:
    print("Khong tim thay luat nao thoa man dieu kien voi nguong loc (0.8 <= lift <= 1.25).")

## L. FP-Growth và so sánh Apriori

In [38]:
from mlxtend.frequent_patterns import fpgrowth
import time

# TODO 18:
# Chạy fpgrowth cùng MIN_SUPPORT
start_time_fp = time.time()
fpgrowth_itemsets = fpgrowth(basket, min_support=MIN_SUPPORT, use_colnames=True)
runtime_fpgrowth = time.time() - start_time_fp

print(f"Thoi gian chay FP-Growth: {runtime_fpgrowth:.4f} giay")
print(f"Thoi gian chay Apriori: {runtime_apriori:.4f} giay")
print(f"So luong tap pho bien tim duoc bang FP-Growth: {len(fpgrowth_itemsets)}")
print(f"So luong tap pho bien tim duoc bang Apriori: {len(apriori_itemsets)}")

# Kiem tra tinh tuong duong cua 2 thuat toan
apriori_sets = set(frozenset(sorted(x)) for x in apriori_itemsets['itemsets'])
fpgrowth_sets = set(frozenset(sorted(x)) for x in fpgrowth_itemsets['itemsets'])
print(f"Hai tap ket qua co hoan toan tuong duong khong? {apriori_sets == fpgrowth_sets}")

## M. Pattern Stability theo thời gian

In [39]:
import pandas as pd
from mlxtend.frequent_patterns import apriori, association_rules

def generate_rules(freq_itemsets, basket_df, min_conf=0.5):
    try:
        return association_rules(
            freq_itemsets,
            num_itemsets=len(basket_df),
            metric="confidence",
            min_threshold=min_conf
        )
    except TypeError:
        return association_rules(
            freq_itemsets,
            metric="confidence",
            min_threshold=min_conf
        )

def normalize_itemset(s):
    return tuple(sorted(list(s)))

def rules_signature(df_rules):
    if df_rules is None or len(df_rules) == 0:
        return set()
    return set(
        (normalize_itemset(a), normalize_itemset(c))
        for a, c in zip(df_rules["antecedents"], df_rules["consequents"])
    )

clean_df['InvoiceDate'] = pd.to_datetime(clean_df['InvoiceDate'])
country_df = clean_df[clean_df['Country'] == 'France'].copy()

# Chia 2 giai doan Period A va B
date_split = pd.to_datetime("2011-06-01")
df_a = country_df[country_df['InvoiceDate'] < date_split]
df_b = country_df[country_df['InvoiceDate'] >= date_split]

basket_a = df_a.groupby(['InvoiceNo', 'Description'])['Quantity'].sum().unstack().fillna(0).map(lambda x: x > 0)
basket_b = df_b.groupby(['InvoiceNo', 'Description'])['Quantity'].sum().unstack().fillna(0).map(lambda x: x > 0)

# Khai thac mau pho bien va luat ket hop
frequent_a = apriori(basket_a, min_support=0.05, use_colnames=True)
frequent_b = apriori(basket_b, min_support=0.05, use_colnames=True)

rules_a = generate_rules(frequent_a, basket_a, min_conf=0.5)
rules_b = generate_rules(frequent_b, basket_b, min_conf=0.5)

sig_a = rules_signature(rules_a)
sig_b = rules_signature(rules_b)

common_rules = sig_a.intersection(sig_b)
print(f"So luat trong Giai doan A: {len(sig_a)}")
print(f"So luat trong Giai doan B: {len(sig_b)}")
print(f"So luat chung giua 2 giai doan: {len(common_rules)}")

## N. Pattern Stability theo thị trường

In [40]:
from mlxtend.frequent_patterns import apriori, association_rules

# So sanh France voi Germany
df_ger = clean_df[clean_df['Country'] == 'Germany']
basket_ger = df_ger.groupby(['InvoiceNo', 'Description'])['Quantity'].sum().unstack().fillna(0).map(lambda x: x > 0)

frequent_ger = apriori(basket_ger, min_support=0.05, use_colnames=True)
rules_ger = generate_rules(frequent_ger, basket_ger, min_conf=0.5)

sig_france = rules_signature(rules_a) # dung du lieu tu Phap da co o buoc truoc
sig_germany = rules_signature(rules_ger)

common_market = sig_france.intersection(sig_germany)
print(f"So luong luat tai Phap: {len(sig_france)}")
print(f"So luong luat tai Duc: {len(sig_germany)}")
print(f"So luong luat chung giua 2 thi truong: {len(common_market)}")

## O. Revenue-aware Analysis

In [41]:
# TODO 21:
clean_df['Revenue'] = clean_df['Quantity'] * clean_df['UnitPrice']

print("--- TOP 10 PRODUCTS BY REVENUE ---")
top_rev = clean_df.groupby('Description')['Revenue'].sum().sort_values(ascending=False).head(10)
print(top_rev)

print("\n--- TOP 10 PRODUCTS BY QUANTITY ---")
top_qty = clean_df.groupby('Description')['Quantity'].sum().sort_values(ascending=False).head(10)
print(top_qty)

## P. Visualization

In [42]:
import matplotlib.pyplot as plt
from mlxtend.frequent_patterns import apriori

# Sinh luat tong quat tu basket
all_frequent = apriori(basket, min_support=0.05, use_colnames=True)
all_rules = generate_rules(all_frequent, basket, min_conf=0.5)

plt.figure(figsize=(12, 5))

# Bieu do 1: Support vs Confidence
plt.subplot(1, 2, 1)
plt.scatter(all_rules['support'], all_rules['confidence'], alpha=0.6, color='blue', edgecolors='k')
plt.xlabel('Support')
plt.ylabel('Confidence')
plt.title('Support vs Confidence')
plt.grid(True)

# Bieu do 2: Confidence vs Lift
plt.subplot(1, 2, 2)
plt.scatter(all_rules['confidence'], all_rules['lift'], alpha=0.6, color='red', edgecolors='k')
plt.xlabel('Confidence')
plt.ylabel('Lift')
plt.title('Confidence vs Lift')
plt.grid(True)

plt.tight_layout()
plt.show()

## Q. Chọn 5 luật tốt nhất
**TODO:** lọc bằng nhiều tiêu chí, sau đó diễn giải từng luật: support, confidence, lift, ý nghĩa và business action.

In [43]:
# Lua chon 5 luat tieu bieu co do ho tro va tin cay phu hop va in ra kem theo chi so phan tich
top_rules = all_rules.sort_values(by=['lift', 'confidence'], ascending=False).head(5)
print(top_rules[['antecedents', 'consequents', 'support', 'confidence', 'lift']])

### Diễn giải 5 luật được chọn

- **Luật 1:** `(SET/6 RED SPOTTY PAPER PLATES) => (SET/6 RED SPOTTY PAPER CUPS)`
  - *Chỉ số:* Support cao, Confidence rất cao, Lift rất lớn.
  - *Ý nghĩa:* Khách hàng khi mua đĩa giấy chấm đỏ thường có xu hướng mua luôn cả cốc giấy chấm đỏ đi kèm.
  - *Hành động:* Sắp xếp đĩa và cốc giấy gần nhau hoặc đóng gói combo cho các bữa tiệc sinh nhật.

- **Luật 2:** `(SET/6 RED SPOTTY PAPER PLATES) => (SET/20 RED RETROSPOT PAPER NAPKINS)`
  - *Chỉ số:* Thể hiện sự liên kết chặt chẽ giữa đĩa ăn và khăn giấy ăn cùng tông màu trang trí.
  - *Hành động:* Khuyến khích mua chéo (cross-selling) thông qua giảm giá khi mua trọn bộ đĩa + khăn giấy.

- **Luật 3:** `(PLASTERS IN TIN CIRCUS PARADE) => (PLASTERS IN TIN WOODLAND ANIMALS)`
  - *Chỉ số:* Các loại hộp băng dán hoạt hình thường được mua kèm cùng nhau.
  - *Hành động:* Bán combo nhiều hộp băng dán hoạt hình khác nhau với giá ưu đãi.

- **Luật 4:** `(ALARM CLOCK BAKELIKE PINK) => (ALARM CLOCK BAKELIKE GREEN)`
  - *Chỉ số:* Thể hiện xu hướng mua nhiều màu sắc khác nhau của cùng một loại đồng hồ báo thức cổ điển.
  - *Hành động:* Trưng bày nhiều màu sắc kế cạnh nhau trên kệ hàng và website.

- **Luật 5:** `(RED RETROSPOT MINI CASES) => (POSTAGE)`
  - *Chỉ số:* Khách mua vali mini chấm đỏ thường đi kèm phí bưu điện để vận chuyển.
  - *Hành động:* Nên tích hợp ưu đãi miễn phí vận chuyển khi giá trị đơn hàng đạt một ngưỡng nhất định nhằm kích cầu mua thêm.

In [44]:
import pandas as pd

# Xuất bảng luật kết hợp tổng quát ra CSV
all_rules_export = all_rules.copy()
# Chuyển đổi kiểu dữ liệu frozenset sang chuỗi để dễ đọc và tương thích tốt khi mở bằng Excel
all_rules_export['antecedents'] = all_rules_export['antecedents'].apply(lambda x: ', '.join(list(x)))
all_rules_export['consequents'] = all_rules_export['consequents'].apply(lambda x: ', '.join(list(x)))
all_rules_export.to_csv('online_retail_association_rules.csv', index=False, encoding='utf-8-sig')

# Xuất bảng tập phổ biến ra CSV
all_frequent_export = all_frequent.copy()
all_frequent_export['itemsets'] = all_frequent_export['itemsets'].apply(lambda x: ', '.join(list(x)))
all_frequent_export.to_csv('online_retail_frequent_itemsets.csv', index=False, encoding='utf-8-sig')

print("Đã xuất dữ liệu thành công:")
print("- Tệp luật kết hợp: online_retail_association_rules.csv")
print("- Tệp tập phổ biến: online_retail_frequent_itemsets.csv")

## R. Research Question

- **RQ của nhóm:** Sự kết hợp của các nhóm sản phẩm (Frequent Patterns) có biến động đáng kể giữa các thị trường quốc gia khác nhau hay không (Ví dụ Pháp và Đức)?
- **Thiết kế thí nghiệm dự kiến:** Sử dụng cùng một ngưỡng `min_support` và `min_confidence` để chạy khai thác luật kết hợp trên dữ liệu giao dịch của Pháp và Đức. Đo lường tỷ lệ các luật kết hợp trùng nhau trên tổng số luật của từng thị trường để đánh giá sự tương đồng và khác biệt văn hóa tiêu dùng giữa hai quốc gia.

## Kết luận cuối buổi

Qua quá trình thực hành, Em nhận thấy các thuật toán khai thác luật kết hợp như Apriori và FP-Growth cho kết quả tập phổ biến hoàn toàn tương đương nhau, tuy nhiên FP-Growth tối ưu hóa vượt trội về mặt thời gian chạy khi kích thước dữ liệu tăng và ngưỡng hỗ trợ giảm thấp. Việc phân tích sự ổn định theo thời gian và quốc gia chỉ ra rằng hành vi mua sắm của khách hàng có tính biến động lớn và chịu ảnh hưởng trực tiếp bởi thị trường địa phương. Điểm giới hạn của phân tích hiện tại là chỉ tập trung vào sự xuất hiện của mặt hàng (biến nhị phân) mà chưa tối ưu hóa việc phân bổ ngân sách dựa trên giá trị doanh thu thực tế mang lại từ mỗi sản phẩm.